# Binary Classification From Scratch — ML Foundations

This notebook consolidates the concepts covered in our binary-classification journey, using **NumPy and Matplotlib first**, without sklearn.

### Concepts covered
- Supervised learning, samples, features and labels
- Linear score: $z = Xw+b$
- Sigmoid and probabilities
- Binary cross-entropy
- Gradient derivation and gradient descent
- Full logistic-regression training
- Loss and decision-boundary visualization
- Accuracy, confusion matrix, precision, recall and F1
- Thresholds, TPR/FPR, ROC and AUC
- XOR and linear inseparability
- Single neuron and hidden-layer forward propagation
- Backpropagation for a 2→3→1 neural network
- ReLU, mini-batch concepts, train/validation/test, overfitting and L2 regularization
- Final XOR solution using a small MLP

**Learning principle:** understand → implement → visualize → experiment.

## 1. Environment

Google Colab runs notebooks in the browser and lets us combine explanatory text, equations, executable Python and visualizations in one document. This notebook intentionally uses only lightweight libraries for the from-scratch portion.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
print('NumPy:', np.__version__)

## 2. Binary-classification dataset

Each row is one sample. Each column is one feature.

$X\in\mathbb{R}^{N\times d}$ and $y\in\{0,1\}^N$.

For this small example, the two classes are linearly separable.

In [ ]:
X = np.array([
    [1, 1],
    [1.5, 1],
    [2, 1.5],
    [2.5, 2],
    [7, 7],
    [8, 7],
    [8, 8],
    [9, 8]
], dtype=float)

y = np.array([0, 0, 0, 0, 1, 1, 1, 1])

print('X shape:', X.shape)
print('y shape:', y.shape)
print('X:\n', X)
print('y:', y)

## 3. Linear model and decision boundary

The model first computes a score:

$z=Xw+b$

The decision boundary is:

$w_1x_1+w_2x_2+b=0$.

In [ ]:
w = np.array([1.0, 1.0])
b = -5.0

z = X @ w + b
y_pred_linear = (z >= 0).astype(int)

print('Scores:', z)
print('Predictions:', y_pred_linear)

## 4. Sigmoid: score → probability

$\sigma(z)=\frac{1}{1+e^{-z}}$

The sigmoid maps any real-valued score into $(0,1)$.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z_test = np.array([-10, -2, 0, 2, 10], dtype=float)
print('z:', z_test)
print('sigmoid(z):', sigmoid(z_test))

p = sigmoid(z)
print('\nProbabilities:', p)
print('Predictions at threshold 0.5:', (p >= 0.5).astype(int))

## 5. Binary cross-entropy

$L=-[y\log p+(1-y)\log(1-p)]$

For a dataset, we minimize the mean loss. Clipping prevents numerical problems with $\log(0)$.

In [ ]:
def binary_cross_entropy(y_true, p):
    eps = 1e-15
    p = np.clip(p, eps, 1 - eps)
    return -np.mean(y_true * np.log(p) + (1 - y_true) * np.log(1 - p))

print('BCE:', binary_cross_entropy(y, p))

## 6. Gradient from the chain rule

For sigmoid + binary cross-entropy:

$\frac{\partial L}{\partial z}=p-y$

Therefore:

$dw=\frac{1}{N}X^T(p-y)$

$db=\mathrm{mean}(p-y)$

This is the seed of backpropagation.

In [ ]:
error = p - y
dw = (X.T @ error) / len(X)
db = np.mean(error)

print('error:', error)
print('dw:', dw)
print('db:', db)

## 7. One gradient-descent update

$w\leftarrow w-\eta dw$

$b\leftarrow b-\eta db$

In [ ]:
w_demo = np.array([0.0, 0.0])
b_demo = 0.0
eta = 0.1

p_before = sigmoid(X @ w_demo + b_demo)
loss_before = binary_cross_entropy(y, p_before)

error = p_before - y
dw = (X.T @ error) / len(X)
db = np.mean(error)

w_demo -= eta * dw
b_demo -= eta * db

p_after = sigmoid(X @ w_demo + b_demo)
loss_after = binary_cross_entropy(y, p_after)

print('Loss before:', loss_before)
print('Loss after :', loss_after)
print('Updated w:', w_demo)
print('Updated b:', b_demo)

## 8. Train logistic regression from scratch

In [ ]:
w = np.zeros(2)
b = 0.0
learning_rate = 0.1
epochs = 1000
loss_history = []

for epoch in range(epochs):
    z = X @ w + b
    p = sigmoid(z)
    loss = binary_cross_entropy(y, p)
    loss_history.append(loss)

    error = p - y
    dw = (X.T @ error) / len(X)
    db = np.mean(error)

    w -= learning_rate * dw
    b -= learning_rate * db

print('Final w:', w)
print('Final b:', b)
print('Final loss:', loss_history[-1])

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(loss_history)
plt.xlabel('Epoch')
plt.ylabel('Binary Cross-Entropy Loss')
plt.title('Logistic Regression: Loss vs Epoch')
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(X[y == 0, 0], X[y == 0, 1], label='Class 0')
plt.scatter(X[y == 1, 0], X[y == 1, 1], label='Class 1')

x1 = np.linspace(0, 10, 100)
x2 = -(w[0] * x1 + b) / w[1]
plt.plot(x1, x2, label='Decision boundary')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Learned Decision Boundary')
plt.legend()
plt.grid(True)
plt.show()

## 9. Evaluation from scratch

Definitions:

- $TP$: true positive
- $TN$: true negative
- $FP$: false positive
- $FN$: false negative

$Accuracy=\frac{TP+TN}{TP+TN+FP+FN}$

$Precision=\frac{TP}{TP+FP}$

$Recall=\frac{TP}{TP+FN}$

$F1=\frac{2PR}{P+R}$

In [ ]:
p = sigmoid(X @ w + b)
y_pred = (p >= 0.5).astype(int)

TP = np.sum((y == 1) & (y_pred == 1))
TN = np.sum((y == 0) & (y_pred == 0))
FP = np.sum((y == 0) & (y_pred == 1))
FN = np.sum((y == 1) & (y_pred == 0))

accuracy = (TP + TN) / len(y)
precision = TP / (TP + FP) if TP + FP else 0
recall = TP / (TP + FN) if TP + FN else 0
f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0

print('Confusion matrix:')
print(np.array([[TN, FP], [FN, TP]]))
print('\nAccuracy :', accuracy)
print('Precision:', precision)
print('Recall   :', recall)
print('F1       :', f1)

## 10. Thresholds, ROC and AUC

$TPR=\frac{TP}{TP+FN}$

$FPR=\frac{FP}{FP+TN}$

The ROC curve plots TPR against FPR as the classification threshold changes. AUC is the numerical area under that curve.

In [ ]:
thresholds = np.linspace(1, 0, 201)
tpr_values = []
fpr_values = []

for threshold in thresholds:
    pred = (p >= threshold).astype(int)
    TP = np.sum((y == 1) & (pred == 1))
    TN = np.sum((y == 0) & (pred == 0))
    FP = np.sum((y == 0) & (pred == 1))
    FN = np.sum((y == 1) & (pred == 0))
    tpr_values.append(TP / (TP + FN) if TP + FN else 0)
    fpr_values.append(FP / (FP + TN) if FP + TN else 0)

fpr_values = np.array(fpr_values)
tpr_values = np.array(tpr_values)

# Sort by FPR so numerical integration is well-defined.
order = np.argsort(fpr_values)
fpr_sorted = fpr_values[order]
tpr_sorted = tpr_values[order]
auc = np.trapezoid(tpr_sorted, fpr_sorted)

print('AUC:', auc)

plt.figure(figsize=(7, 6))
plt.plot(fpr_sorted, tpr_sorted, label=f'ROC (AUC={auc:.3f})')
plt.plot([0, 1], [0, 1], '--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.grid(True)
plt.show()

## 11. XOR: deliberately break the linear classifier

XOR is:

$[0,0]\to0$, $[0,1]\to1$, $[1,0]\to1$, $[1,1]\to0$.

No single straight line can separate these classes. With symmetric zero initialization, this particular dataset can also produce zero gradients. Use a small nonzero initialization to expose the representation limitation rather than the initialization artifact.

In [ ]:
X_xor = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
y_xor = np.array([0,1,1,0])

w = np.array([0.1, -0.1])
b = 0.0
lr = 0.1
epochs = 5000
xor_loss = []

for epoch in range(epochs):
    p_xor = sigmoid(X_xor @ w + b)
    loss = binary_cross_entropy(y_xor, p_xor)
    xor_loss.append(loss)

    error = p_xor - y_xor
    dw = (X_xor.T @ error) / len(X_xor)
    db = np.mean(error)
    w -= lr * dw
    b -= lr * db

p_xor = sigmoid(X_xor @ w + b)
pred_xor = (p_xor >= 0.5).astype(int)

print('Final XOR probabilities:', p_xor)
print('Final XOR predictions  :', pred_xor)
print('XOR accuracy            :', np.mean(pred_xor == y_xor))

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(xor_loss)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('XOR: Logistic Regression Cannot Represent the Pattern')
plt.grid(True)
plt.show()

## 12. Single neuron

A logistic-regression classifier is essentially one sigmoid neuron:

$z=w^Tx+b$, then $a=\sigma(z)$.

A neural network becomes more expressive by composing many neurons and nonlinear layers.

In [ ]:
x = np.array([[2.0], [3.0]])
w_neuron = np.array([[0.5], [0.8]])
b_neuron = np.array([[0.2]])

z = w_neuron.T @ x + b_neuron
a = sigmoid(z)

print('z =', z)
print('a =', a)

## 13. Two-layer network: forward propagation

For a 2→3→1 network:

$z^{(1)}=W_1x+b_1$

$a^{(1)}=\sigma(z^{(1)})$

$z^{(2)}=W_2a^{(1)}+b_2$

$\hat y=\sigma(z^{(2)})$

In [ ]:
x = np.array([[2.0], [3.0]])
W1 = np.array([[0.5, 0.8], [0.2, -0.4], [1.0, 0.3]])
b1 = np.array([[0.2], [0.1], [-0.5]])
W2 = np.array([[0.7, -0.5, 0.9]])
b2 = np.array([[0.1]])

z1 = W1 @ x + b1
a1 = sigmoid(z1)
z2 = W2 @ a1 + b2
y_hat = sigmoid(z2)

print('z1 =\n', z1)
print('a1 =\n', a1)
print('z2 =\n', z2)
print('y_hat =\n', y_hat)

## 14. Backpropagation

For sigmoid + BCE at the output:

$\delta_2=\hat y-y$

$dW_2=\delta_2(a^{(1)})^T$

$db_2=\delta_2$

Then propagate backward:

$\delta_1=(W_2^T\delta_2)\odot a^{(1)}(1-a^{(1)})$

$dW_1=\delta_1x^T$

$db_1=\delta_1$

Backpropagation is the chain rule applied from the output toward the input.

## 15. Complete MLP: solve XOR

Now we combine forward propagation, BCE, backpropagation and gradient descent.

In [ ]:
np.random.seed(42)

X = X_xor
y = y_xor.reshape(-1, 1).astype(float)

W1 = np.random.randn(2, 3) * 0.5
b1 = np.zeros((1, 3))
W2 = np.random.randn(3, 1) * 0.5
b2 = np.zeros((1, 1))

lr = 1.0
epochs = 10000
loss_history = []

for epoch in range(epochs):
    # Forward
    z1 = X @ W1 + b1
    a1 = sigmoid(z1)
    z2 = a1 @ W2 + b2
    y_hat = sigmoid(z2)

    # Loss
    loss = binary_cross_entropy(y, y_hat)
    loss_history.append(loss)

    # Backprop
    dz2 = y_hat - y
    dW2 = (a1.T @ dz2) / len(X)
    db2 = np.mean(dz2, axis=0, keepdims=True)

    dz1 = (dz2 @ W2.T) * a1 * (1 - a1)
    dW1 = (X.T @ dz1) / len(X)
    db1 = np.mean(dz1, axis=0, keepdims=True)

    # Update
    W2 -= lr * dW2
    b2 -= lr * db2
    W1 -= lr * dW1
    b1 -= lr * db1

# Final inference
a1 = sigmoid(X @ W1 + b1)
y_hat = sigmoid(a1 @ W2 + b2)
y_pred = (y_hat >= 0.5).astype(int)

print('XOR results')
for i in range(len(X)):
    print(f'{X[i]} -> probability={y_hat[i,0]:.4f}, prediction={y_pred[i,0]}, actual={int(y[i,0])}')

print('Accuracy:', np.mean(y_pred == y))

plt.figure(figsize=(8, 5))
plt.plot(loss_history)
plt.xlabel('Epoch')
plt.ylabel('Binary Cross-Entropy Loss')
plt.title('MLP Learning XOR')
plt.grid(True)
plt.show()

## 16. ReLU and a deeper MLP

For hidden layers, ReLU is often preferred:

$ReLU(z)=\max(0,z)$

$ReLU'(z)=1$ for $z>0$ and $0$ for $z<0$.

Typical binary-classification architecture:

$X\rightarrow Linear\rightarrow ReLU\rightarrow Linear\rightarrow ReLU\rightarrow Linear\rightarrow Sigmoid$.

## 17. Mini-batch, validation and generalization

**Mini-batch:** update parameters using a subset of samples rather than the whole dataset.

**Epoch:** one complete pass through the training set.

**Train/validation/test:** training updates weights; validation guides model/hyperparameter choices; test estimates final generalization.

**Overfitting:** training performance is much better than unseen-data performance.

**L2 regularization:**

$L_{total}=L+\frac{\lambda}{2}\sum_l\|W_l\|_F^2$

and the weight gradient gains a $+\lambda W$ term.

## 18. Experiment checklist

Run the following experiments and record what changes:

1. Logistic regression with learning rates 0.01, 0.1 and 1.0.
2. Logistic regression with 10, 100 and 1000 epochs.
3. XOR with zero initialization vs nonzero initialization.
4. Compare sigmoid hidden layers with ReLU hidden layers.
5. Change the number of hidden neurons: 1, 2, 3, 8.
6. Change the decision threshold from 0.5 to 0.3 and 0.7.
7. Observe how threshold changes precision and recall.
8. Add L2 regularization and compare the training loss.

### Key takeaway

$\boxed{\text{Forward pass}\rightarrow\text{Loss}\rightarrow\text{Backpropagation}\rightarrow\text{Update}}$

This loop is the core training mechanism we will reuse for deeper neural networks, CNNs and eventually ML/VLSI applications.

# Next stage: Multiclass Classification

We will extend binary classification from

$P(y=1)$

to a probability vector

$[P(y=0),P(y=1),...,P(y=K-1)]$

using **Softmax + multiclass cross-entropy**.